This seciton is to prove the profit of the strategy does not come from the market's momentum.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# Section III: Common-factor / lead-lag tests
# - Use Yahoo Finance S&P 500 data already saved in ../data/
# - Build a 6/3 momentum strategy (change K=6 below to match paper's 6/6 rep strategy)
# - Test:
#   C. Serial covariance of 6-month market returns
#   D. Regression of momentum profits on lagged squared market returns
# ============================================================

# ----------------------------
# 1) Load and prepare data
# ----------------------------
prices = pd.read_parquet("../data/sp500_prices.parquet")

daily_close = prices["Close"].copy().sort_index()
daily_close = daily_close.dropna(axis=1, how="all")

monthly_close = daily_close.resample("ME").last()
monthly_close = monthly_close.dropna(axis=1, how="all")

monthly_ret = monthly_close.pct_change(fill_method=None)

# Equal-weighted market proxy from available S&P 500 constituents
market_ret = monthly_ret.mean(axis=1, skipna=True).rename("market_ret")

print("monthly_close shape:", monthly_close.shape)
print("monthly_ret shape:", monthly_ret.shape)
print("market_ret shape:", market_ret.shape)
print("monthly_close index head:", monthly_close.index[:5])
print("monthly_ret index head:", monthly_ret.index[:5])


# ----------------------------
# 2) Helper functions
# ----------------------------
def compute_formation_signal(monthly_close: pd.DataFrame, J: int = 6) -> pd.DataFrame:
    """
    Past J-month cumulative return used for ranking.
    signal[t] = P_t / P_{t-J} - 1
    """
    return monthly_close.pct_change(periods=J, fill_method=None)


def assign_deciles(signal_row: pd.Series, n_deciles: int = 10) -> pd.Series:
    """
    1 = losers decile
    10 = winners decile
    """
    x = signal_row.dropna()
    if len(x) < n_deciles:
        return pd.Series(dtype="int64")

    # qcut on ranks avoids duplicate-edge issues
    ranks = x.rank(method="first")
    labels = pd.qcut(ranks, n_deciles, labels=False) + 1
    return pd.Series(labels.astype(int), index=x.index)


def momentum_backtest(
    monthly_close: pd.DataFrame,
    monthly_ret: pd.DataFrame,
    J: int = 6,
    K: int = 3,
    n_deciles: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Overlapping momentum backtest:
    - each month rank stocks by past J-month cumulative return
    - winners = top decile, losers = bottom decile
    - hold each subportfolio for K months
    - at each month, average across all active subportfolios
    """
    signal = compute_formation_signal(monthly_close, J=J)
    dates = monthly_close.index

    subportfolios = []
    holdings_rows = []

    for formation_pos, dt in enumerate(dates):
        if dt not in signal.index:
            continue

        row = signal.loc[dt]
        deciles = assign_deciles(row, n_deciles=n_deciles)
        if deciles.empty:
            continue

        winners = deciles[deciles == n_deciles].index.tolist()
        losers = deciles[deciles == 1].index.tolist()

        start_pos = formation_pos + 1
        end_pos = min(formation_pos + K, len(dates) - 1)
        if start_pos >= len(dates):
            continue

        subportfolios.append(
            {
                "formation_date": dt,
                "start_pos": start_pos,
                "end_pos": end_pos,
                "winners": winners,
                "losers": losers,
            }
        )

        holdings_rows.append(
            {
                "formation_date": dt,
                "n_winners": len(winners),
                "n_losers": len(losers),
                "winners": winners,
                "losers": losers,
            }
        )

    out = []
    for t_pos, dt in enumerate(dates):
        active_winner_rets = []
        active_loser_rets = []

        r = monthly_ret.loc[dt]

        for sp in subportfolios:
            if sp["start_pos"] <= t_pos <= sp["end_pos"]:
                w = sp["winners"]
                l = sp["losers"]

                wret = r[w].dropna().mean() if len(w) > 0 else np.nan
                lret = r[l].dropna().mean() if len(l) > 0 else np.nan

                if pd.notna(wret) and pd.notna(lret):
                    active_winner_rets.append(wret)
                    active_loser_rets.append(lret)

        if len(active_winner_rets) == 0:
            continue

        winner_ret = float(np.mean(active_winner_rets))
        loser_ret = float(np.mean(active_loser_rets))
        ls_ret = winner_ret - loser_ret

        out.append(
            {
                "date": dt,
                "winner_ret": winner_ret,
                "loser_ret": loser_ret,
                "long_short_ret": ls_ret,
                "n_active_subportfolios": len(active_winner_rets),
            }
        )

    strategy_df = pd.DataFrame(out).set_index("date").sort_index()
    holdings_df = pd.DataFrame(holdings_rows).set_index("formation_date").sort_index()
    return strategy_df, holdings_df


def performance_summary(r: pd.Series, periods_per_year: int = 12, nw_lags: int = 5) -> pd.Series:
    """
    Summary stats for a monthly return series.
    """
    r = r.dropna()
    if len(r) == 0:
        return pd.Series(dtype=float)

    mean_m = r.mean()
    vol_m = r.std(ddof=1)
    ann_ret = (1 + mean_m) ** periods_per_year - 1
    ann_vol = vol_m * np.sqrt(periods_per_year)
    sharpe = (mean_m / vol_m) * np.sqrt(periods_per_year) if vol_m > 0 else np.nan

    X = np.ones((len(r), 1))
    nw = sm.OLS(r.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

    return pd.Series(
        {
            "mean_monthly": mean_m,
            "ann_return_approx": ann_ret,
            "ann_vol": ann_vol,
            "sharpe": sharpe,
            "nw_tstat_mean": nw.tvalues[0],
            "nw_pvalue_mean": nw.pvalues[0],
            "n_obs": len(r),
        }
    )


def ols_hac(y: pd.Series, x: pd.Series, maxlags: int = 5) -> sm.regression.linear_model.RegressionResultsWrapper:
    """
    OLS with Newey-West / HAC standard errors.
    """
    df = pd.concat([y.rename("y"), x.rename("x")], axis=1).dropna()
    X = sm.add_constant(df["x"])
    return sm.OLS(df["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})


# ----------------------------
# 3) Run the momentum strategy (6/3)
# ----------------------------
J = 6
K = 3   # change this to 6 if you want the paper-style 6/6 representative strategy

strategy_df, holdings_df = momentum_backtest(
    monthly_close=monthly_close,
    monthly_ret=monthly_ret,
    J=J,
    K=K,
    n_deciles=10,
)

print("\nMomentum strategy head:")
display(strategy_df.head())

print("\nMomentum strategy summary:")
display(performance_summary(strategy_df["long_short_ret"]).round(4))

print("\nAverage winner / loser / long-short return:")
display(strategy_df[["winner_ret", "loser_ret", "long_short_ret"]].mean().to_frame("mean_return").round(4))


# ----------------------------
# 4) Section III C:
#    Serial covariance of 6-month market returns
# ----------------------------
market_6m = ((1 + market_ret).rolling(6).apply(np.prod, raw=True) - 1).rename("market_6m")

market_6m = market_6m.dropna()
market_6m_lag1 = market_6m.shift(1)

market_6m_serial_cov = market_6m.cov(market_6m_lag1)
market_6m_serial_corr = market_6m.autocorr(1)

print("\n=== Section III C: Market return serial dependence ===")
print(f"6-month market serial covariance: {market_6m_serial_cov:.6f}")
print(f"6-month market autocorrelation:   {market_6m_serial_corr:.6f}")

# Optional: monthly market autocov/autocorr for comparison
market_monthly_cov = market_ret.cov(market_ret.shift(1))
market_monthly_corr = market_ret.autocorr(1)

print(f"Monthly market serial covariance:  {market_monthly_cov:.6f}")
print(f"Monthly market autocorrelation:    {market_monthly_corr:.6f}")


# ----------------------------
# 5) Section III D:
#    Regress momentum profits on lagged squared market returns
# ----------------------------
# Use the 6-month cumulative market return over t-6 ... t-1 as the factor proxy,
# aligned to month t by shifting one month forward.
lagged_market_6m = market_6m.shift(1).rename("lagged_market_6m")

reg_df = pd.concat(
    [
        strategy_df["long_short_ret"].rename("ls_ret"),
        lagged_market_6m,
    ],
    axis=1,
).dropna()

# Demean the lagged factor proxy before squaring, following the paper's logic
reg_df["lagged_market_6m_demeaned"] = reg_df["lagged_market_6m"] - reg_df["lagged_market_6m"].mean()
reg_df["lagged_market_6m_demeaned_sq"] = reg_df["lagged_market_6m_demeaned"] ** 2

model = ols_hac(
    y=reg_df["ls_ret"],
    x=reg_df["lagged_market_6m_demeaned_sq"],
    maxlags=5,
)

print("\n=== Section III D: Momentum profit vs lagged squared market return ===")
print(model.summary())

print("\nKey coefficients:")
print(f"alpha  = {model.params['const']:.6f}")
print(f"theta  = {model.params['x']:.6f}")
print(f"t(alpha) = {model.tvalues['const']:.4f}")
print(f"t(theta) = {model.tvalues['x']:.4f}")
print(f"p(theta) = {model.pvalues['x']:.6f}")


# ----------------------------
# 6) Compact comparison table
# ----------------------------
section3_summary = pd.DataFrame(
    {
        "metric": [
            "market_6m_serial_cov",
            "market_6m_serial_corr",
            "market_monthly_cov",
            "market_monthly_corr",
            "momentum_mean_monthly",
            "momentum_nw_tstat",
            "reg_alpha",
            "reg_theta_on_lagged_market_sq",
            "reg_theta_tstat",
            "reg_theta_pvalue",
            "n_obs_reg",
        ],
        "value": [
            market_6m_serial_cov,
            market_6m_serial_corr,
            market_monthly_cov,
            market_monthly_corr,
            performance_summary(strategy_df["long_short_ret"])["mean_monthly"],
            performance_summary(strategy_df["long_short_ret"])["nw_tstat_mean"],
            model.params["const"],
            model.params["x"],
            model.tvalues["x"],
            model.pvalues["x"],
            int(model.nobs),
        ],
    }
)

print("\n=== Section III summary table ===")
display(section3_summary)

# ----------------------------
# 7) Optional save
# ----------------------------
Path("../results").mkdir(exist_ok=True)
strategy_df.to_csv("../results/section3_momentum_63.csv")
holdings_df.to_csv("../results/section3_momentum_63_holdings.csv")
reg_df.to_csv("../results/section3_regression_frame.csv")
section3_summary.to_csv("../results/section3_summary.csv", index=False)

print("\nSaved outputs to ../results/")

monthly_close shape: (132, 502)
monthly_ret shape: (132, 502)
market_ret shape: (132,)
monthly_close index head: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')
monthly_ret index head: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')

Momentum strategy head:


,winner_ret,loser_ret,long_short_ret,n_active_subportfolios
date,,,,
2015-08-31,-0.050526,-0.064526,0.014000,1
2015-09-30,-0.023500,-0.052382,0.028881,2
2015-10-31,0.053361,0.107357,-0.053996,3
2015-11-30,0.017957,-0.003790,0.021748,3
2015-12-31,-0.009801,-0.062108,0.052306,3



Momentum strategy summary:


mean_monthly           0.0036
ann_return_approx      0.0443
ann_vol                0.1711
sharpe                 0.2535
nw_tstat_mean          1.2433
nw_pvalue_mean         0.2138
n_obs                125.0000
dtype: float64


Average winner / loser / long-short return:


,mean_return
winner_ret,0.0202
loser_ret,0.0166
long_short_ret,0.0036



=== Section III C: Market return serial dependence ===
6-month market serial covariance: 0.006609
6-month market autocorrelation:   0.711381
Monthly market serial covariance:  -0.000352
Monthly market autocorrelation:    -0.158895

=== Section III D: Momentum profit vs lagged squared market return ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.8448
Date:                Sat, 27 Jun 2026   Prob (F-statistic):              0.360
Time:                        16:16:56   Log-Likelihood:                 199.63
No. Observations:                 125   AIC:                            -395.3
Df Residuals:                     123   BIC:                            -389.6
Df Model:                           1                           

,metric,value
0,market_6m_serial_cov,0.006609
1,market_6m_serial_corr,0.711381
2,market_monthly_cov,-0.000352
3,market_monthly_corr,-0.158895
4,momentum_mean_monthly,0.003615
5,momentum_nw_tstat,1.243259
6,reg_alpha,0.006158
7,reg_theta_on_lagged_market_sq,-0.275960
8,reg_theta_tstat,-0.919107
9,reg_theta_pvalue,0.358040



Saved outputs to ../results/


Our objective in this section is to examine whether the profitability of the momentum strategy can be explained by the dynamics of common market factors rather than delayed reactions to firm-specific information.

Following Jegadeesh and Titman (1993), we first examine the serial dependence of the market factor. Using an equal-weighted S&P 500 market proxy, we find that the monthly market return exhibits a slightly negative first-order autocorrelation (-0.159), consistent with mild mean reversion. However, the six-month cumulative market return displays strong positive serial dependence, with a serial covariance of 0.0066 and an autocorrelation of 0.711. This differs from the original paper, where the six-month market serial covariance is negative, implying that common-factor persistence cannot explain momentum profits. Our result suggests that the modern equity market may exhibit stronger medium-term persistence than the historical CRSP sample used in the paper.

We next replicate the lead-lag test proposed by the authors. Specifically, we regress the momentum portfolio return on the squared demeaned lagged six-month market return. The estimated coefficient is negative (-0.276) but statistically insignificant (t = -0.92, p = 0.36). Therefore, we do not find evidence supporting the hypothesis that momentum profits are generated by delayed reactions to common market shocks. This finding is directionally consistent with Jegadeesh and Titman (1993), who also report a negative coefficient and conclude that lead-lag effects are unlikely to be the primary source of momentum profits.

Overall, our results suggest that although medium-term market persistence appears stronger in the modern S&P 500 sample than in the original CRSP sample, the profitability of the momentum strategy cannot be fully attributed to common-factor dynamics. Consequently, the evidence continues to favor the interpretation that momentum is more closely related to delayed reactions to firm-specific information rather than systematic market-wide effects.